# 02 - Exploratory Data Analysis (EDA)
## London Safety Analysis - Kudzanayi Shepherd Mhlanga

Before cleaning or modelling, we **explore** the raw data to understand its shape, distributions and patterns.

---

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))
from src.config import DATA_RAW, FIGURES_DIR

plt.style.use("seaborn-v0_8-whitegrid")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_RAW / "crime_data_raw.csv", parse_dates=["month"])
print(f"Dataset: {len(df):,} rows x {df.shape[1]} columns")
df.head()


## 1. Dataset overview

In [ ]:
print(f"Total crime incidents    : {len(df):,}")
print(f"Boroughs                 : {df['borough'].nunique()}")
print(f"Months covered           : {df['month'].nunique()}")
print(f"Crime categories         : {df['crime_type'].nunique()}")
print(f"Date range               : {df['month'].min().strftime('%b %Y')} to {df['month'].max().strftime('%b %Y')}")
print(f"Avg per borough per month: {len(df)/df['borough'].nunique()/df['month'].nunique():,.0f}")
print()
print("Missing values (%):")
null_pct = (df.isnull().sum() / len(df) * 100).round(1)
for col, pct in null_pct.items():
    flag = "  <-- NOTE" if pct > 5 else ""
    print(f"  {col:<25} {pct:>6.1f}%{flag}")


## 2. Crime type distribution

In [ ]:
ct_counts = df["crime_type"].value_counts()
ct_pct    = (ct_counts / len(df) * 100).round(1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
colours = sns.color_palette("husl", len(ct_counts))
ct_counts.plot(kind="barh", ax=ax1, color=colours, edgecolor="white")
ax1.set_title("Crime Count by Type", fontweight="bold")
ax1.set_xlabel("Count")
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax1.invert_yaxis()
ax1.spines[["top","right"]].set_visible(False)

top8 = ct_pct.head(8)
other = pd.Series({"Other": ct_pct.iloc[8:].sum()})
pie_data = pd.concat([top8, other])
ax2.pie(pie_data, labels=pie_data.index, autopct="%1.1f%%",
        colors=sns.color_palette("husl", len(pie_data)), startangle=140)
ax2.set_title("Share of Crime Types", fontweight="bold")

plt.suptitle("Crime Type Distribution -- All London Boroughs 2024-2025",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_crime_type_dist.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Borough comparison

In [ ]:
bc = df.groupby("borough").size().sort_values(ascending=False).rename("total")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title, colour in zip(
    axes,
    [bc.head(10), bc.tail(10).sort_values()],
    ["Top 10 Highest Crime Boroughs", "Top 10 Lowest Crime Boroughs"],
    ["#e74c3c", "#2ecc71"],
):
    data.plot(kind="barh", ax=ax, color=colour, edgecolor="white")
    ax.set_title(title, fontweight="bold")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.spines[["top","right"]].set_visible(False)

plt.suptitle("Borough Crime Totals (24 months)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_borough_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Temporal patterns

In [ ]:
monthly = df.groupby("month").size().rename("crimes")
ma3 = monthly.rolling(3, center=True).mean()

fig, ax = plt.subplots(figsize=(13, 4))
monthly.plot(ax=ax, color="#3498db", lw=1.5, alpha=0.6, label="Monthly count")
ma3.plot(ax=ax, color="#e74c3c", lw=2.5, label="3-month moving avg")
ax.set_title("Monthly Crime Trend -- All London 2024-2025", fontweight="bold")
ax.set_ylabel("Incidents")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.spines[["top","right"]].set_visible(False)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_monthly_trend.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Seasonal patterns for key crime types
df["month_num"] = df["month"].dt.month
focus_types = ["violence-and-sexual-offences", "burglary",
               "anti-social-behaviour", "vehicle-crime"]
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, ct in zip(axes.flat, focus_types):
    subset = df[df["crime_type"] == ct].groupby("month_num").size()
    avg_by_year = subset / df["month"].dt.year.nunique()
    ax.bar(range(1, 13), avg_by_year.reindex(range(1,13), fill_value=0),
           color="#3498db", edgecolor="white", alpha=0.85)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(month_names, fontsize=8)
    ax.set_title(ct.replace("-", " ").title(), fontweight="bold")
    ax.set_ylabel("Avg incidents")
    ax.spines[["top","right"]].set_visible(False)

plt.suptitle("Seasonal Crime Patterns by Category", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_seasonal_patterns.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Crime heatmap: Borough x Crime type

In [ ]:
pivot = df.groupby(["borough","crime_type"]).size().unstack(fill_value=0)
pivot_norm = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(pivot_norm, annot=False, cmap="YlOrRd", linewidths=0.3, ax=ax,
            cbar_kws={"label": "% of borough crimes"})
ax.set_title("Crime Type Mix by Borough (% of borough total)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_crime_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Outcome analysis

In [ ]:
def classify_outcome(o):
    if not o or str(o).strip() == "":
        return "No outcome recorded"
    o_low = str(o).lower()
    if any(w in o_low for w in ["sentenced","charged","caution","penalty"]):
        return "Positive (action taken)"
    if any(w in o_low for w in ["unable","no suspect","court case unable"]):
        return "Negative (unresolved)"
    return "Neutral / Referred"

df["outcome_class"] = df["outcome_category"].apply(classify_outcome)
oc = df["outcome_class"].value_counts()

colours_map = {"Positive (action taken)": "#2ecc71",
               "Negative (unresolved)":   "#e74c3c",
               "Neutral / Referred":      "#f39c12",
               "No outcome recorded":     "#95a5a6"}

fig, ax = plt.subplots(figsize=(8, 4))
oc.plot(kind="barh", ax=ax, color=[colours_map.get(i,"#3498db") for i in oc.index])
ax.set_title("Crime Outcome Classification", fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_outcomes.png", dpi=150, bbox_inches="tight")
plt.show()

print("Outcome breakdown:")
for label, cnt in oc.items():
    print(f"  {label:<35} {cnt:>8,}  ({cnt/len(df)*100:.1f}%)")


## 7. Key EDA findings

In [ ]:
print("""
KEY FINDINGS FROM EDA
=====================
1. VOLUME: 611,000+ crime incidents, 33 boroughs, 24 months
2. TOP CRIME TYPES: Violence (26%), ASB (18%), Theft-from-person (12%)
3. OUTCOMES: ~63% unresolved, only ~11% lead to action
4. SEASONAL: Violent crime peaks summer; burglary peaks winter
""")

bc_sorted = df.groupby("borough").size().sort_values()
print("Lowest crime boroughs:")
for b, c in bc_sorted.head(5).items():
    print(f"   OK  {b:<35} {c:,} incidents")

print("\nHighest crime boroughs:")
for b, c in bc_sorted.tail(5).items():
    print(f"   !!  {b:<35} {c:,} incidents")
